In [0]:
%run "../common/define_path"

In [0]:
tracking_raw_path
tracking_br_path

In [0]:
def ingest_raw_to_bronze( 
    source_path: str, 
    target_table: str, 
    schema_path: str, 
    checkpoint_path: str, 
    file_format: str = "csv" 
): 
    from pyspark.sql.functions import current_timestamp
    df = ( 
        spark.readStream 
            .format("cloudFiles") 
            .option("cloudFiles.format", file_format) 
            .option("cloudFiles.schemaLocation", schema_path) 
            .option("cloudFiles.inferColumnTypes", "true") 
            .option('header','true')
            .load(source_path) 
            .withColumn("ingestion_time", current_timestamp()) 
            
    ) 
 
    ( 
        df.writeStream 
          .format("delta") 
          .option("checkpointLocation", checkpoint_path) 
          .trigger(availableNow=True) 
          .table(target_table) 
    ) 

In [0]:
# Bronze object configuration is externalized to a Unity Catalog control table.
# Relative paths are resolved against the environment base paths from common/define_path.
config_table = "dev.control.bronze_ingestion_config"

config_rows = (
    spark.table(config_table)
         .filter("is_active = true")
         .orderBy("load_order")
         .collect()
)

if not config_rows:
    raise ValueError(f"No active Bronze ingestion configuration found in {config_table}")

tables = [
  {
    "source": f"{tracking_raw_path}/{r['source_rel_path']}",
    "target": r["target_table"],
    "schema": f"{tracking_br_path}/{r['schema_rel_path']}",
    "checkpoint": f"{tracking_br_path}/{r['checkpoint_rel_path']}",
    "format": r["file_format"]
  }
  for r in config_rows
]

for t in tables: 
    ingest_raw_to_bronze( 
        source_path=t["source"], 
        target_table=t["target"], 
        schema_path=t["schema"],  
        checkpoint_path=t["checkpoint"],
        file_format=t["format"] 
    )